In [1]:
import numpy as np
import pandas as pd
import yaml
from sqlalchemy import create_engine

#### Conexion con las bases de datos

In [3]:
with open('../config.yml', 'r') as f:
    config = yaml.safe_load(f)
    config_co = config['mensajeria_bd']
    config_etl = config['etl_mensajeria']
    
url_co = (f"{config_co['drivername']}://{config_co['user']}:{config_co['password']}@{config_co['host']}:"
          f"{config_co['port']}/{config_co['dbname']}")
url_etl = (f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:"
           f"{config_etl['port']}/{config_etl['dbname']}")

engine_origen = create_engine(url_co)
engine_destino = create_engine(url_etl)

In [4]:
novedad = pd.read_sql_table("mensajeria_novedadesservicio", engine_origen)
tipo_novedad = pd.read_sql_table("mensajeria_tiponovedad", engine_origen)

In [5]:
novedad.head()

,id,fecha_novedad,tipo_novedad_id,descripcion,servicio_id,es_prueba,mensajero_id
0,4,2023-11-30 05:00:00+00:00,1,A,51,True,7
1,5,2023-11-30 05:00:00+00:00,1,Halo,51,True,7
2,6,2023-11-30 05:00:00+00:00,1,A,51,True,7
3,7,2023-11-30 05:00:00+00:00,1,B,51,True,7
4,8,2023-11-30 05:00:00+00:00,1,A,51,True,7


In [6]:
tipo_novedad.head()

,id,nombre
0,2,No puedo continuar
1,1,Novedades del servicio


In [7]:
dim_novedad = novedad.merge(tipo_novedad, left_on='tipo_novedad_id', right_on='id', how='left')

In [8]:
dim_novedad.head()

,id_x,fecha_novedad,tipo_novedad_id,descripcion,servicio_id,es_prueba,mensajero_id,id_y,nombre
0,4,2023-11-30 05:00:00+00:00,1,A,51,True,7,1,Novedades del servicio
1,5,2023-11-30 05:00:00+00:00,1,Halo,51,True,7,1,Novedades del servicio
2,6,2023-11-30 05:00:00+00:00,1,A,51,True,7,1,Novedades del servicio
3,7,2023-11-30 05:00:00+00:00,1,B,51,True,7,1,Novedades del servicio
4,8,2023-11-30 05:00:00+00:00,1,A,51,True,7,1,Novedades del servicio


In [9]:
dim_novedad.rename(columns={
    'id_x': 'id_novedad',
    'nombre': 'tipo_novedad'
}, inplace=True)

In [10]:
dim_novedad.columns.tolist()

['id_novedad',
 'fecha_novedad',
 'tipo_novedad_id',
 'descripcion',
 'servicio_id',
 'es_prueba',
 'mensajero_id',
 'id_y',
 'tipo_novedad']

In [11]:
columnas_necesarias = ["id_novedad", "tipo_novedad", "descripcion"]
dim_novedad = dim_novedad[columnas_necesarias].copy()
dim_novedad.head()

,id_novedad,tipo_novedad,descripcion
0,4,Novedades del servicio,A
1,5,Novedades del servicio,Halo
2,6,Novedades del servicio,A
3,7,Novedades del servicio,B
4,8,Novedades del servicio,A


In [12]:
dim_novedad.to_sql('dim_novedad',con=engine_destino,if_exists='replace',index_label='key_dim_novedad')

208